# Panel con los tags de la mentora

El notebook arma el panel: una fila por empresa y trimestre calendario, una columna por concepto. Usa solo los tags de la mentora (`docs/professor-materials/sec-edgar-guide.docx`, sección 4). `concepts.py` y nuestras construcciones no se usan (reunión de grupo, 2026-10-27).

Pasos:

1. Elegir las empresas sobrevivientes: una presentación en cada uno de los 24 trimestres 2018–2023.
2. Quedarse con los tags de la mentora.
3. Calcular los flujos trimestrales.
4. Asignar cada valor a su trimestre calendario.
5. Elegir un tag por empresa, concepto y trimestre.
6. Quedarse con las empresas con los 9 conceptos en los 24 trimestres.
7. Guardar `data/panel.csv`.

Entradas: `data/sub.csv` y `data/num.csv` (salen de `build_dataset.ipynb`).

In [1]:
import sys
from pathlib import Path

import pandas as pd


if "google.colab" in sys.modules:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")

    CANDIDATOS = [
        Path("/content/drive/MyDrive/Colab Notebooks/Diplodatos MAI Grupo 6"),
        Path("/content/drive/MyDrive/Diplodatos MAI Grupo 6"),
    ]
    WORKING_DIR = next((p for p in CANDIDATOS if p.exists()), None)
    if WORKING_DIR is None:
        raise FileNotFoundError(
            "No encuentro la carpeta del grupo. ¿Agregaste el acceso directo "
            "en 'Mi unidad' desde 'Compartido conmigo'?"
        )
else:
    WORKING_DIR = Path.cwd()


## Paso 1: sobrevivientes

Una empresa sobrevive si presentó un documento (10-K o 10-Q) con `period` en cada uno de los 24 trimestres 2018Q1–2023Q4. Es el criterio de `survivorship_bias.ipynb`.

`quarter()` asigna una fecha a su trimestre calendario con la regla de `docs/decisions.md`, sección 11: el trimestre que contiene más meses del trimestre fiscal. En código: restar un mes y tomar el trimestre. Por ejemplo, un cierre el 31 de enero de 2019 va a 2018Q4.

`survivorship_bias.ipynb` toma el trimestre de `period` sin restar el mes. Las dos reglas dan los mismos 143 sobrevivientes.

Cada empresa tiene un solo sector y un solo nombre: los de su documento más reciente por `filed` (`docs/decisions.md`, sección 2).

In [ ]:
def quarter(month: pd.Series) -> pd.Series:
    """Calendar quarter with most months of the fiscal quarter that ends in `month` (decisions.md, section 11)."""
    return (month - 1).dt.asfreq("Q")


sub = pd.read_csv(f"{WORKING_DIR}/data/sub.csv", dtype=str)
sub["month"] = pd.to_datetime(sub["period"], format="%Y%m%d").dt.to_period("M")
sub["quarter"] = quarter(sub["month"])
sub = sub[sub["quarter"].between(pd.Period("2018Q1"), pd.Period("2023Q4"))]

quarters = sub.groupby("cik")["quarter"].nunique()
survivors = quarters[quarters == 24].index

# name and sector of the most recent document (`filed` is AAAAMMDD, so text order = date order)
companies = (
    sub.sort_values("filed").groupby("cik")[["name", "sector"]].last().loc[survivors]
)
companies["sector"].value_counts()

sector
energy    101
health     42
Name: count, dtype: int64

## Paso 2: tags de la mentora

`MENTOR` copia la tabla de la guía para los 9 conceptos de los 7 ratios (`docs/ratios.md`). El orden de cada lista es la prioridad: primero el tag principal, después los alternativos.

- **Flujos** (`FLOWS`): miden un período, por ejemplo los ingresos de un trimestre. Tienen `qtrs` de 1 a 4.
- **Saldos**: miden una fecha, por ejemplo los activos al cierre. Tienen `qtrs` = 0.

La celda lee de `num.csv` solo las filas de los sobrevivientes con un tag de la mentora.

In [3]:
MENTOR = {
    "revenue": [
        "Revenues",
        "SalesRevenueNet",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
    ],
    "operating_income": ["OperatingIncomeLoss"],
    "net_income": ["NetIncomeLoss", "ProfitLoss"],
    "assets": ["Assets"],
    "liabilities": ["Liabilities"],
    "equity": ["StockholdersEquity", "StockholdersEquityAttributableToParent"],
    "current_assets": ["AssetsCurrent"],
    "current_liabilities": ["LiabilitiesCurrent"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities"],
}
FLOWS = ["revenue", "operating_income", "net_income", "operating_cash_flow"]

# tag -> concept, and tag -> position in its list (0 = principal tag)
CONCEPT = {t: c for c, tags in MENTOR.items() for t in tags}
RANK = {t: i for tags in MENTOR.values() for i, t in enumerate(tags)}

num = pd.read_csv(
    f"{WORKING_DIR}/data/num.csv",
    dtype={"cik": str, "ddate": str},
    usecols=["cik", "tag", "ddate", "qtrs", "value"],
)
num = num[num["cik"].isin(survivors) & num["tag"].isin(CONCEPT)]
num["concept"] = num["tag"].map(CONCEPT)
num["month"] = pd.to_datetime(num["ddate"], format="%Y%m%d").dt.to_period("M")
num.groupby("qtrs").size()

qtrs
0    19653
1    13687
2     4809
3     4778
4     5168
dtype: int64

## Paso 3: flujos trimestrales

El panel usa el flujo de un trimestre (`docs/decisions.md`, sección 10). Muchos documentos no lo traen: traen el acumulado desde el inicio del año fiscal.

- Si hay valor con `qtrs` = 1, se usa ese valor.
- Si no, se resta: acumulado de k trimestres menos acumulado de k − 1 trimestres que termina 3 meses antes. Los dos acumulados empiezan en el mismo mes.

Ejemplos:

- **Cuarto trimestre:** año (`qtrs` = 4, diciembre) menos 9 meses (`qtrs` = 3, septiembre). El 10-K no trae el cuarto trimestre solo.
- **Flujo operativo del segundo trimestre:** 6 meses (`qtrs` = 2, junio) menos 3 meses (`qtrs` = 1, marzo). Los 10-Q suelen traer este flujo solo acumulado.

La resta se hace con el mismo tag (`docs/decisions.md`, sección 12, regla A). Nunca mezcla dos tags.

La celda mueve cada acumulado de k − 1 trimestres 3 meses hacia adelante y lo une a la fila del acumulado de k trimestres. Después resta. La tabla cuenta los valores trimestrales según su origen: `qtrs` = 1 es un valor reportado; 2, 3 y 4 son restas.

In [4]:
flows = num[num["concept"].isin(FLOWS)]

# the accumulated k-1 quarters, moved to the row of the accumulated k quarters (3 months later)
previous = flows.assign(month=flows["month"] + 3, qtrs=flows["qtrs"] + 1)
flows = flows.merge(
    previous[["cik", "tag", "month", "qtrs", "value"]],
    on=["cik", "tag", "month", "qtrs"],
    how="left",
    suffixes=("", "_previous"),
)
flows["value"] = flows["value"].where(
    flows["qtrs"] == 1, flows["value"] - flows["value_previous"]
)

# ponytail: "3 months before" compares months, so a 52/53-week year ending on the 1st of a month
# can miss its previous accumulated value; compare days if that matters
flows = flows.dropna(subset="value")
flows.groupby("qtrs").size()

qtrs
1    13687
2     4701
3     4706
4     4120
dtype: int64

## Verificación de la resta

Algunas fechas tienen el trimestre reportado (`qtrs` = 1) y también la resta de acumulados. La celda compara los dos valores. Si coinciden, la resta es correcta.

La celda mide tres tolerancias:

- **1 USD:** el valor es idéntico.
- **1.000 USD:** muchas empresas redondean a miles, así que la resta puede diferir en el último dígito.
- **1%:** una diferencia mayor indica otro problema, por ejemplo un trimestre anterior corregido en un documento posterior. El panel usa la primera versión publicada de cada número (`docs/decisions.md`, sección 8).

In [ ]:
reported = flows[flows["qtrs"] == 1]
subtracted = flows[flows["qtrs"] > 1]
both = reported.merge(
    subtracted, on=["cik", "tag", "month"], suffixes=("_reported", "_subtracted")
)
difference = (both["value_reported"] - both["value_subtracted"]).abs()
print(f"fechas con los dos valores: {len(both)}")
print(f"diferencia <= 1 USD: {(difference <= 1).mean():.1%}")
print(f"diferencia <= 1.000 USD: {(difference <= 1000).mean():.1%}")
print(
    f"diferencia <= 1%: {(difference <= 0.01 * both['value_reported'].abs()).mean():.1%}"
)

fechas con los dos valores: 7824
diferencia <= 1 USD: 93.1%
diferencia <= 1.000 USD: 96.2%
diferencia <= 1%: 97.4%


## Paso 4: trimestre calendario

Cada valor va al trimestre de su fecha (`quarter()`, paso 1). Antes se eliminan las fechas que no son el cierre de un documento de la empresa (`sub.period`). Dos casos:

- **Comparativos con otro calendario.** Sustainable Projects cambió su cierre de mayo a diciembre en 2019. Sus documentos nuevos traen comparativos con cierre en diciembre de 2018, y el trimestre 2018Q4 ya tiene el cierre de noviembre.
- **Subperíodos por quiebra.** Una empresa que sale de una quiebra (por ejemplo, Gulfport Energy en 2021) reporta el período antes y después de la salida por separado.

En los dos casos hay dos fechas en el mismo trimestre. El panel usa la fecha del documento de ese trimestre. Cada sobreviviente tiene un documento en los 24 trimestres, así que el filtro no deja trimestres vacíos.

`docs/decisions.md`, sección 11, usa otro filtro: elimina los meses fuera del calendario fiscal de la empresa (módulo 3). Ese filtro no resuelve Sustainable Projects: tiene dos calendarios.

Una fecha puede tener el trimestre reportado y una o más restas. Se usa el reportado; si no hay, la resta con menos trimestres acumulados.

El `assert` comprueba que no quedan dos fechas en el mismo trimestre.

In [6]:
# a date can have the reported quarter and one or more subtractions: keep the smallest `qtrs` (reported first)
flows = flows.sort_values("qtrs").drop_duplicates(["cik", "tag", "month"])
stocks = num[~num["concept"].isin(FLOWS) & (num["qtrs"] == 0)]
values = pd.concat([flows, stocks])[["cik", "tag", "concept", "month", "value"]]

# keep only the closing dates of the company's documents (one per quarter for a survivor)
values = values.merge(sub[["cik", "month"]].drop_duplicates(), on=["cik", "month"])
values["quarter"] = quarter(values["month"])

assert not values.duplicated(["cik", "tag", "quarter"]).any()
values

,cik,tag,concept,month,value,quarter
0,1457919,NetIncomeLoss,net_income,2020-03,-7.800000e+04,2020Q1
1,1282648,NetCashProvidedByUsedInOperatingActivities,operating_cash_flow,2020-03,1.234300e+07,2020Q1
2,1282648,RevenueFromContractWithCustomerExcludingAssess...,revenue,2020-03,4.702400e+07,2020Q1
3,1315061,NetIncomeLoss,net_income,2020-03,3.990000e+05,2020Q1
4,1409947,Revenues,revenue,2020-03,1.760000e+06,2020Q1
...,...,...,...,...,...,...
30702,1293818,StockholdersEquity,equity,2023-12,-1.156158e+07,2023Q4
30703,1293818,Liabilities,liabilities,2023-12,1.344386e+07,2023Q4
30704,1293818,Assets,assets,2023-12,1.882274e+06,2023Q4
30705,1477449,Liabilities,liabilities,2023-12,2.066296e+09,2023Q4


## Paso 5: un tag por empresa, concepto y trimestre

En cada empresa, concepto y trimestre se usa el primer tag de `MENTOR` con valor (regla A, `docs/decisions.md`, sección 12). Una empresa puede cambiar de tag entre trimestres.

`panel` tiene una fila por empresa y trimestre, y una columna por concepto. NaN significa que ningún tag de la mentora tiene valor.

In [7]:
panel = (
    # principal tag first: sort by position in the list, keep the first row of each concept and quarter
    values.sort_values("tag", key=lambda t: t.map(RANK))
    .drop_duplicates(["cik", "concept", "quarter"])
    .pivot(index=["cik", "quarter"], columns="concept", values="value")[list(MENTOR)]
)
panel

concept             revenue  operating_income  net_income      assets  \
cik     quarter                                                         
1001614 2018Q1    1367000.0          118000.0   1243000.0   9408000.0   
        2018Q2    1475000.0           97000.0    109000.0   9718000.0   
        2018Q3    1654000.0          285000.0    298000.0  10098000.0   
        2018Q4    1375000.0         -260000.0    -81000.0   9484000.0   
        2019Q1    1171000.0         -138000.0    -96000.0   9525000.0   
...                     ...               ...         ...         ...   
96793   2022Q4   14392000.0         2040000.0   1951000.0  26169000.0   
        2023Q1   11533000.0         -774000.0   -762000.0  24430000.0   
        2023Q2   10987000.0        -1975000.0  -1426000.0  23269000.0   
        2023Q3    8555000.0         -450000.0  -1344000.0  21723000.0   
        2023Q4    8510000.0         -433000.0  -3075000.0  18787000.0   

concept          liabilities     equity  current_assets  current_liabilities  \
cik     quarter                                                                
1001614 2018Q1     2986000.0  6422000.0       4341000.0             591000.0   
        2018Q2     3187000.0  6531000.0       4598000.0             751000.0   
        2018Q3     3269000.0  6829000.0       4916000.0             830000.0   
        2018Q4     2717000.0  6767000.0       4347000.0             548000.0   
        2019Q1     2850000.0  6675000.0       4481000.0             659000.0   
...                      ...        ...             ...                  ...   
96793   2022Q4           NaN        NaN      15024000.0            5633000.0   
        2023Q1           NaN        NaN      13304000.0            4677000.0   
        2023Q2           NaN        NaN      12274000.0            4869000.0   
        2023Q3           NaN        NaN      16531000.0            4993000.0   
        2023Q4           NaN        NaN      13710000.0            5228000.0   

concept          operating_cash_flow  
cik     quarter                       
1001614 2018Q1              138000.0  
        2018Q2              408000.0  
        2018Q3              517000.0  
        2018Q4              274000.0  
        2019Q1              -18000.0  
...                              ...  
96793   2022Q4              546000.0  
        2023Q1              606000.0  
        2023Q2             -421000.0  
        2023Q3            -1137000.0  
        2023Q4             -393000.0  

[3432 rows x 9 columns]

## Paso 6: empresas con los 9 conceptos en los 24 trimestres

Entran al panel las empresas con todos los conceptos en los 24 trimestres. Es un panel balanceado (`docs/decisions.md`, sección 12).

La tabla cuenta, por sector, los sobrevivientes con cada concepto en los 24 trimestres. La fila `all` son las empresas con los 9 conceptos.

In [8]:
full = panel.notna().groupby(level="cik").sum().reindex(survivors, fill_value=0).eq(24)
full["all"] = full.all(axis=1)
full.groupby(companies["sector"]).sum().T

sector,energy,health
concept,,
revenue,74,29
operating_income,82,36
net_income,87,41
assets,100,42
liabilities,67,34
equity,85,41
current_assets,99,42
current_liabilities,100,42
operating_cash_flow,87,41


## Empresas que quedan afuera por pocos valores

La tabla lista los sobrevivientes fuera del panel a los que les faltan 4 valores o menos. Un valor es un concepto en un trimestre. La columna `missing` dice qué concepto falta y en qué trimestre.

In [ ]:
missing = panel.isna().stack()
missing = missing[missing].reset_index()[["cik", "quarter", "concept"]]
few = missing.groupby("cik").filter(lambda m: len(m) <= 4)
few["missing"] = few["concept"] + " " + few["quarter"].astype(str)
few.groupby("cik")["missing"].agg(", ".join).to_frame().join(companies).sort_values(
    ["sector", "name"]
)

,missing,name,sector
cik,,,
1342936,liabilities 2018Q3,"ADVANCED VOICE RECOGNITION SYSTEMS, INC",energy
793306,operating_cash_flow 2018Q4,BLUE DOLPHIN ENERGY CO,energy
1486159,"revenue 2020Q4, operating_income 2020Q4, net_i...",CHORD ENERGY CORP,energy
352955,liabilities 2023Q1,"CKX LANDS, INC.",energy
23194,"revenue 2018Q1, operating_cash_flow 2018Q1, op...",COMSTOCK RESOURCES INC,energy
858470,net_income 2018Q4,COTERRA ENERGY INC.,energy
1539838,"operating_cash_flow 2018Q1, operating_cash_flo...","DIAMONDBACK ENERGY, INC.",energy
874499,"revenue 2021Q2, operating_income 2021Q2, net_i...",GULFPORT ENERGY CORP,energy
866829,"operating_cash_flow 2018Q1, operating_cash_flo...",HELIX ENERGY SOLUTIONS GROUP INC,energy


## Paso 7: guardar

`data/panel.csv` tiene una fila por empresa y trimestre: `cik`, `name`, `sector`, `quarter` y los 9 conceptos, en USD. El `assert` comprueba que el panel está balanceado: 24 trimestres por empresa y ningún valor vacío.

In [10]:
keep = full.index[full["all"]]
out = panel.loc[keep].reset_index().merge(companies, left_on="cik", right_index=True)
out = out[["cik", "name", "sector", "quarter", *MENTOR]]

assert len(out) == 24 * len(keep) and out.notna().all().all()
out.to_csv(f"{WORKING_DIR}/data/panel.csv", index=False)
out.groupby("sector")["cik"].nunique()

sector
energy    23
health    21
Name: cik, dtype: int64

## Resultado

- **Paso 1.** Sobreviven 143 empresas: 101 de energía y 42 de salud.
- **Paso 3.** La resta de acumulados coincide con el trimestre reportado en el 96,2% de las fechas con 1.000 USD de tolerancia, y en el 97,4% con 1% de tolerancia.
- **Paso 6.** El panel tiene 44 empresas: 23 de energía y 21 de salud. Son 1.056 filas (44 × 24). Las dos cantidades están dentro del rango de la consigna: 20 a 30 por sector.
- **Concepto que más limita.** En energía, pasivo (67 de 101) y ingresos (74 de 101). En salud, ingresos (29 de 42).

`mentor_concepts.ipynb` (opción 1) cuenta 33 empresas de energía y 21 de salud. Esa cuenta acepta cualquier valor del trimestre, también un acumulado. El panel necesita el valor del trimestre solo, así que energía baja de 33 a 23.

Causas en las empresas que quedan afuera por pocos valores (cell-17):

- **Cambio de tag entre el 10-Q y el 10-K.** Los 7 fondos Ridgewood reportan la ganancia de 9 meses de 2023 con `NetIncomeLoss` y la del año con `ProfitLoss`. La regla A no resta dos tags distintos, así que falta 2023Q4.
- **Quiebra.** Chord Energy (antes Oasis Petroleum, 2020Q4) y Gulfport Energy (2021Q2) reportan el trimestre de la salida de la quiebra en dos subperíodos. Ninguno es un trimestre completo.

PRECAUCIÓN: el panel tiene sesgo de supervivencia. Solo entran las empresas con presentación en los 24 trimestres (`survivorship_bias.ipynb`).

In [11]:
out

,cik,name,sector,quarter,revenue,operating_income,net_income,assets,liabilities,equity,current_assets,current_liabilities,operating_cash_flow
0,1006655,EVOLUTION PETROLEUM CORP,energy,2018Q1,1.024957e+07,3663267.0,3068354.0,9.168633e+07,1.548855e+07,7.619778e+07,3.181943e+07,3.415181e+06,5417958.0
1,1006655,EVOLUTION PETROLEUM CORP,energy,2018Q2,1.142686e+07,5182663.0,4532750.0,9.366254e+07,1.637306e+07,7.728948e+07,3.214756e+07,4.430214e+06,6021032.0
2,1006655,EVOLUTION PETROLEUM CORP,energy,2018Q3,1.230708e+07,5994927.0,5795801.0,9.681801e+07,1.692314e+07,7.989488e+07,3.411750e+07,4.649453e+06,7656012.0
3,1006655,EVOLUTION PETROLEUM CORP,energy,2018Q4,1.104812e+07,4733747.0,3904565.0,9.641986e+07,1.573407e+07,8.068580e+07,3.402958e+07,3.204691e+06,6488243.0
4,1006655,EVOLUTION PETROLEUM CORP,energy,2019Q1,9.501028e+06,2952955.0,2398875.0,9.549152e+07,1.551685e+07,7.997467e+07,3.391727e+07,2.794703e+06,4188897.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1051,927066,DAVITA INC.,health,2022Q4,2.916888e+09,256252000.0,68103000.0,1.692825e+10,1.470345e+10,7.123260e+08,3.155292e+09,2.619747e+09,343722000.0
1052,927066,DAVITA INC.,health,2023Q1,2.872699e+09,311645000.0,115547000.0,1.672779e+10,1.430908e+10,8.254700e+08,3.039739e+09,2.565093e+09,462557000.0
1053,927066,DAVITA INC.,health,2023Q2,3.000369e+09,405291000.0,178691000.0,1.695884e+10,1.432923e+10,1.017444e+09,3.001968e+09,2.403418e+09,450107000.0
1054,927066,DAVITA INC.,health,2023Q3,3.121332e+09,495939000.0,246632000.0,1.692874e+10,1.410406e+10,1.199201e+09,3.135359e+09,2.537930e+09,661162000.0
